# LongMemEval A0.5 Oracle + Diagnostic Benchmark

This notebook runs the next action after A0:

1. Locate or import A0 artifacts from `results/`.
2. Create an answerable-only oracle retrieval log from `answer_session_ids`.
3. Run the same reader on oracle evidence sessions.
4. Evaluate oracle hypotheses with the same LLM judge.
5. Compare A0 BM25 session@5 against A0.5 oracle evidence.
6. Build diagnostic artifacts for retrieval misses, reader failures, failed-case taxonomy, completion length, and prompt leakage checks.

Expected A0 artifacts, if available:

- `A0_bm25_session_lme_s_cleaned_hypotheses.jsonl`
- `A0_bm25_session_lme_s_cleaned_eval_log.jsonl`
- `A0_bm25_session_lme_s_cleaned_retrieval_log.jsonl`
- `A0_bm25_session_lme_s_cleaned_summary.json`
- `A0_bm25_session_lme_s_cleaned_failed_cases.csv`
- `A0_bm25_session_lme_s_cleaned_cost_latency.csv`

If you run this in a fresh Kaggle notebook, attach or extract the A0 output archive first, or set `A0_ARTIFACT_DIR` to the folder containing the A0 files.


In [1]:
from pathlib import Path
import csv
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from collections import Counter, defaultdict

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PROJECT_DIR_NAME = os.getenv('PROJECT_DIR_NAME', 'LongMemEval-Experiment')
REPO_URL = os.getenv('LONGMEMEVAL_REPO', 'https://github.com/toanthangO20/LongMemEval-Experiment.git')
CHECKOUT_DIR = os.getenv('LONGMEMEVAL_CHECKOUT_DIR', PROJECT_DIR_NAME)

DATASET_NAME = os.getenv('LONGMEMEVAL_DATASET', 'longmemeval_s_cleaned.json')
A0_EXPERIMENT_ID = os.getenv('A0_EXPERIMENT_ID', 'A0_bm25_session_lme_s_cleaned')
A05_EXPERIMENT_ID = os.getenv('A05_EXPERIMENT_ID', 'A05_oracle_diagnostic_lme_s_cleaned')

# Set these to 0 if you only want to prepare diagnostics from existing files.
RUN_ORACLE_GENERATION = os.getenv('RUN_ORACLE_GENERATION', '1') == '1'
RUN_ORACLE_EVALUATION = os.getenv('RUN_ORACLE_EVALUATION', '1') == '1'

# Reader/judge configuration. Keep this aligned with A0 for a clean oracle gap.
DEFAULT_OPENAI_BASE_URL = os.getenv('DEFAULT_OPENAI_BASE_URL', 'https://splashed-nastily-stopped.ngrok-free.dev/v1')
GEN_MODEL_NAME = os.getenv('GEN_MODEL_NAME', 'cx/gpt-5.2')
GEN_MODEL_ALIAS = os.getenv('GEN_MODEL_ALIAS', 'router-gpt-5.2')
METRIC_MODEL_SHORT = os.getenv('METRIC_MODEL_SHORT', 'router-gpt-5.2')
METRIC_MODEL_NAME = os.getenv('METRIC_MODEL_NAME', 'cx/gpt-5.2')
MODEL_MAX_LENGTH = int(os.getenv('MODEL_MAX_LENGTH', '128000'))
GEN_LENGTH = int(os.getenv('GEN_LENGTH', '300'))
HISTORY_FORMAT = os.getenv('HISTORY_FORMAT', 'json')
USERONLY = os.getenv('USERONLY', 'false')
OPENAI_DEFAULT_HEADERS = os.getenv('OPENAI_DEFAULT_HEADERS', '{"ngrok-skip-browser-warning":"true"}')
OPENAI_MAX_RETRIES = int(os.getenv('OPENAI_MAX_RETRIES', '8'))

REPORT_TOPKS = [1, 3, 5, 10, 20]
TOPK_CONTEXT = int(os.getenv('TOPK_CONTEXT', '5'))
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '7'))
MAX_NOTEBOOK_PREVIEW = int(os.getenv('MAX_NOTEBOOK_PREVIEW', '25'))

print('Working root:', ROOT)
print('Dataset:', DATASET_NAME)
print('A0 experiment:', A0_EXPERIMENT_ID)
print('A0.5 experiment:', A05_EXPERIMENT_ID)
print('Run oracle generation:', RUN_ORACLE_GENERATION)
print('Run oracle evaluation:', RUN_ORACLE_EVALUATION)
print('Reader model:', GEN_MODEL_NAME)
print('Judge model:', METRIC_MODEL_NAME)


Working root: /kaggle/working
Dataset: longmemeval_s_cleaned.json
A0 experiment: A0_bm25_session_lme_s_cleaned
A0.5 experiment: A05_oracle_diagnostic_lme_s_cleaned
Run oracle generation: True
Run oracle evaluation: True
Reader model: cx/gpt-5.2
Judge model: cx/gpt-5.2


## Install lightweight dependencies

This notebook reuses the same API-based generation and evaluation path as A0. `httpx==0.27.2` is pinned for `openai==1.35.1` compatibility.


In [2]:
%pip install -q openai==1.35.1 httpx==0.27.2 backoff==2.2.1 rank-bm25==0.2.2 tiktoken==0.7.0 sentence-transformers==2.7.0 scikit-learn tqdm==4.66.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.8/326.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 

In [3]:
import httpx
import openai
print('openai version:', openai.__version__)
print('httpx version:', httpx.__version__)
assert tuple(map(int, httpx.__version__.split('.')[:2])) < (0, 28), 'httpx must be < 0.28 for openai==1.35.1'


openai version: 1.35.1
httpx version: 0.27.2


## Secrets and subprocess helpers

Secrets are read from environment variables first, then Kaggle Secrets. API keys are never passed as command-line arguments.


In [4]:
def run_cmd(cmd, cwd=None, env=None, check=True, log_file=None):
    shown = [str(x) for x in cmd]
    print('$', ' '.join(shown))
    run_kwargs = {
        'cwd': str(cwd) if cwd else None,
        'env': env,
        'check': check,
        'text': True,
    }
    if log_file:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        print('Writing command output to:', log_file)
        with log_file.open('w', encoding='utf-8') as stream:
            try:
                return subprocess.run(cmd, stdout=stream, stderr=subprocess.STDOUT, **run_kwargs)
            except subprocess.CalledProcessError:
                print('Command failed. Last log lines:')
                lines = log_file.read_text(encoding='utf-8', errors='replace').splitlines()
                for line in lines[-80:]:
                    print(line)
                raise
    return subprocess.run(cmd, **run_kwargs)


def load_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ''


OPENAI_API_KEY = load_secret('OPENAI_API_KEY')
OPENAI_ORGANIZATION = load_secret('OPENAI_ORGANIZATION')
OPENAI_BASE_URL = load_secret('OPENAI_BASE_URL') or os.getenv('OPENAI_BASE_URL', DEFAULT_OPENAI_BASE_URL)

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if OPENAI_ORGANIZATION:
    os.environ['OPENAI_ORGANIZATION'] = OPENAI_ORGANIZATION
if OPENAI_BASE_URL:
    os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_DEFAULT_HEADERS'] = OPENAI_DEFAULT_HEADERS
os.environ['TOKENIZER_BACKEND'] = os.getenv('TOKENIZER_BACKEND', 'openai')
os.environ['MODEL_MAX_LENGTH'] = str(MODEL_MAX_LENGTH)
os.environ['METRIC_MODEL_NAME'] = METRIC_MODEL_NAME
os.environ['OPENAI_MAX_RETRIES'] = str(OPENAI_MAX_RETRIES)

print('OPENAI_API_KEY configured:', bool(OPENAI_API_KEY))
print('OPENAI_ORGANIZATION configured:', bool(OPENAI_ORGANIZATION))
print('OPENAI_BASE_URL configured:', bool(OPENAI_BASE_URL))
print('MODEL_MAX_LENGTH:', MODEL_MAX_LENGTH)
print('TOKENIZER_BACKEND:', os.environ['TOKENIZER_BACKEND'])


OPENAI_API_KEY configured: True
OPENAI_ORGANIZATION configured: False
OPENAI_BASE_URL configured: True
MODEL_MAX_LENGTH: 128000
TOKENIZER_BACKEND: openai


## Prepare source tree


In [5]:
def looks_like_longmemeval_repo(path):
    path = Path(path)
    return (path / 'src' / 'retrieval' / 'run_retrieval.py').exists() and (path / 'src' / 'generation' / 'run_generation.py').exists()


candidate_dirs = [Path.cwd(), ROOT / CHECKOUT_DIR, ROOT / 'LongMemEval', ROOT / PROJECT_DIR_NAME]
REPO_DIR = None
for candidate in candidate_dirs:
    if looks_like_longmemeval_repo(candidate):
        REPO_DIR = candidate.resolve()
        break

if REPO_DIR is None:
    REPO_DIR = (ROOT / CHECKOUT_DIR).resolve()
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    if not looks_like_longmemeval_repo(REPO_DIR):
        raise RuntimeError(f'Checkout does not look like a LongMemEval repo: {REPO_DIR}')

RESULTS_DIR = REPO_DIR / 'results'
A05_DIR = RESULTS_DIR / A05_EXPERIMENT_ID
A05_DIR.mkdir(parents=True, exist_ok=True)

print('Using source tree:', REPO_DIR)
print('A0.5 output directory:', A05_DIR)


$ git clone --depth 1 https://github.com/toanthangO20/LongMemEval-Experiment.git /kaggle/working/LongMemEval-Experiment


Cloning into '/kaggle/working/LongMemEval-Experiment'...


Using source tree: /kaggle/working/LongMemEval-Experiment
A0.5 output directory: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned


## Apply Kaggle compatibility patches

These patches are idempotent and keep the benchmark logic intact: no API key in printed args, OpenAI-compatible routers, retries for transient API failures, and NumPy 2.x retrieval metrics compatibility.


In [6]:
def replace_text(path, old, new):
    text = path.read_text(encoding='utf-8')
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        return True
    return False


gen_py = REPO_DIR / 'src' / 'generation' / 'run_generation.py'
gen_text = gen_py.read_text(encoding='utf-8')
if 'import os\n' not in gen_text[:150]:
    gen_text = gen_text.replace('import sys\n', 'import sys\nimport os\n')
gen_py.write_text(gen_text, encoding='utf-8')

replace_text(
    gen_py,
    "    if args.openai_organization:\n        openai.organization = args.openai_organization\n",
    "    openai_organization = args.openai_organization or os.getenv('OPENAI_ORGANIZATION')\n    if openai_organization:\n        openai.organization = openai_organization\n",
)
replace_text(
    gen_py,
    "    parser.add_argument('--openai_key', type=str, required=True)\n",
    "    parser.add_argument('--openai_key', type=str, default=None)\n",
)
replace_text(
    gen_py,
    "def check_args(args):\n    print(args)\n",
    "def check_args(args):\n    safe_args = argparse.Namespace(**vars(args))\n    if safe_args.openai_key:\n        safe_args.openai_key = '***'\n    if safe_args.openai_organization:\n        safe_args.openai_organization = '***'\n    print(safe_args)\n",
)
replace_text(
    gen_py,
    "    client = OpenAI(\n        api_key=args.openai_key,\n        base_url=args.openai_base_url,\n    )",
    "    openai_key = args.openai_key or os.getenv('OPENAI_API_KEY')\n    if not openai_key:\n        raise RuntimeError('OPENAI_API_KEY is required. Set it in the environment or pass --openai_key.')\n    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    client = OpenAI(\n        api_key=openai_key,\n        base_url=args.openai_base_url,\n        default_headers=default_headers,\n    )",
)
replace_text(
    gen_py,
    "    model_max_length = model2maxlength[args.model_name]\n",
    "    model_max_length = model2maxlength.get(args.model_name, int(os.getenv('MODEL_MAX_LENGTH', '128000')))\n",
)
replace_text(
    gen_py,
    "    if 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
    "    if os.getenv('TOKENIZER_BACKEND', '').lower() == 'openai' or 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
)
replace_text(
    gen_py,
    "            total_prompt_tokens += completion.usage.prompt_tokens\n            total_completion_tokens += completion.usage.completion_tokens\n",
    "            usage = getattr(completion, 'usage', None)\n            total_prompt_tokens += (getattr(usage, 'prompt_tokens', 0) or 0)\n            total_completion_tokens += (getattr(usage, 'completion_tokens', 0) or 0)\n",
)
replace_text(
    gen_py,
    "@backoff.on_exception(backoff.constant, (openai.RateLimitError), \n                      interval=5)\n",
    "@backoff.on_exception(\n    backoff.expo,\n    (openai.RateLimitError, openai.APIError, openai.APIConnectionError, openai.APITimeoutError),\n    max_tries=int(os.getenv('OPENAI_MAX_RETRIES', '8')),\n)\n",
)

eval_py = REPO_DIR / 'src' / 'evaluation' / 'evaluate_qa.py'
eval_text = eval_py.read_text(encoding='utf-8')
if 'import os\n' not in eval_text[:150]:
    eval_text = eval_text.replace('import sys\n', 'import sys\nimport os\n')
eval_py.write_text(eval_text, encoding='utf-8')
if "METRIC_MODEL_NAME" not in eval_py.read_text(encoding='utf-8'):
    replace_text(
        eval_py,
        "    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
        "    if metric_model_short not in model_zoo and os.getenv('METRIC_MODEL_NAME'):\n        model_zoo[metric_model_short] = (os.getenv('METRIC_MODEL_NAME'), 'openai')\n    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
    )
replace_text(
    eval_py,
    "        openai_api_base = None\n",
    "        openai_api_base = os.getenv('OPENAI_BASE_URL') or None\n        if not openai_api_key:\n            raise RuntimeError('OPENAI_API_KEY is required for OpenAI-compatible evaluation models.')\n",
)
replace_text(
    eval_py,
    "    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n    )",
    "    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n        default_headers=default_headers,\n    )",
)
replace_text(
    eval_py,
    "@backoff.on_exception(backoff.expo, (openai.RateLimitError,\n                                    openai.APIError))\n",
    "@backoff.on_exception(\n    backoff.expo,\n    (openai.RateLimitError, openai.APIError, openai.APIConnectionError, openai.APITimeoutError),\n    max_tries=int(os.getenv('OPENAI_MAX_RETRIES', '8')),\n)\n",
)
replace_text(eval_py, "            print(json.dumps(entry), file=out_f)\n", "            print(json.dumps(entry), file=out_f, flush=True)\n")

eval_utils_py = REPO_DIR / 'src' / 'retrieval' / 'eval_utils.py'
replace_text(eval_utils_py, 'np.asfarray(relevances)[:k]', 'np.asarray(relevances, dtype=float)[:k]')

print('Compatibility patches applied or already present.')


Compatibility patches applied or already present.


## Load benchmark data


In [7]:
DATA_DIR = REPO_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target_file = DATA_DIR / DATASET_NAME

if not target_file.exists():
    candidates = list(Path('/kaggle/input').rglob(DATASET_NAME)) if Path('/kaggle/input').exists() else []
    if candidates:
        print('Copying dataset from Kaggle input:', candidates[0])
        shutil.copy2(candidates[0], target_file)
    else:
        url = f'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/{DATASET_NAME}'
        print('Downloading:', url)
        urllib.request.urlretrieve(url, target_file)
else:
    print('Dataset already exists:', target_file)

data = json.loads(target_file.read_text(encoding='utf-8'))
ref_by_id = {row['question_id']: row for row in data}
print('Loaded examples:', len(data))
print('Question type counts:')
counts = Counter(row['question_type'] for row in data)
print(json.dumps(dict(sorted(counts.items())), indent=2))


Downloading: https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json
Loaded examples: 500
Question type counts:
{
  "knowledge-update": 78,
  "multi-session": 133,
  "single-session-assistant": 56,
  "single-session-preference": 30,
  "single-session-user": 70,
  "temporal-reasoning": 133
}


## Locate A0 artifacts

This cell looks in common Kaggle locations and also extracts an attached A0 output archive if present.


In [8]:
A0_FILES = {
    'hypotheses': f'{A0_EXPERIMENT_ID}_hypotheses.jsonl',
    'eval_log': f'{A0_EXPERIMENT_ID}_eval_log.jsonl',
    'retrieval_log': f'{A0_EXPERIMENT_ID}_retrieval_log.jsonl',
    'summary': f'{A0_EXPERIMENT_ID}_summary.json',
    'failed_cases': f'{A0_EXPERIMENT_ID}_failed_cases.csv',
    'cost_latency': f'{A0_EXPERIMENT_ID}_cost_latency.csv',
}


def safe_extract_tar(archive_path, dest_dir):
    archive_path = Path(archive_path)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:*') as tar:
        for member in tar.getmembers():
            member_path = (dest_dir / member.name).resolve()
            if not str(member_path).startswith(str(dest_dir.resolve())):
                raise RuntimeError(f'Unsafe tar member path: {member.name}')
        tar.extractall(dest_dir)
    return dest_dir


def maybe_extract_a0_archive():
    archive_patterns = ['*A0*outputs*.tar.gz', 'longmemeval_A0_benchmark_outputs.tar.gz', '*benchmark_outputs.tar.gz']
    roots = [ROOT, REPO_DIR]
    if Path('/kaggle/input').exists():
        roots.append(Path('/kaggle/input'))
    for root in roots:
        for pattern in archive_patterns:
            for archive in root.rglob(pattern):
                dest = ROOT / 'a0_artifacts_extracted'
                print('Extracting possible A0 archive:', archive)
                return safe_extract_tar(archive, dest)
    return None


def candidate_a0_dirs():
    dirs = []
    env_dir = os.getenv('A0_ARTIFACT_DIR')
    if env_dir:
        dirs.append(Path(env_dir))
    dirs.extend([
        REPO_DIR / 'results',
        ROOT / 'results',
        ROOT / 'a0_artifacts_extracted' / 'results',
        ROOT / 'a0_artifacts_extracted' / PROJECT_DIR_NAME / 'results',
    ])
    if Path('/kaggle/input').exists():
        dirs.extend(Path('/kaggle/input').rglob('results'))
    return list(dict.fromkeys(path.resolve() for path in dirs if path.exists()))


maybe_extract_a0_archive()

a0_paths = {}
for folder in candidate_a0_dirs():
    found = {key: folder / filename for key, filename in A0_FILES.items() if (folder / filename).exists()}
    if len(found) >= 3 and {'eval_log', 'retrieval_log', 'summary'}.issubset(found.keys()):
        a0_paths = found
        print('Found A0 artifacts in:', folder)
        break

HAS_A0_ARTIFACTS = bool(a0_paths)
print('HAS_A0_ARTIFACTS:', HAS_A0_ARTIFACTS)
if HAS_A0_ARTIFACTS:
    for key, path in sorted(a0_paths.items()):
        print(f'{key}: {path}')
else:
    print('A0 artifacts were not found. Oracle run can still proceed, but A0 diagnostic CSVs will be skipped.')


HAS_A0_ARTIFACTS: False
A0 artifacts were not found. Oracle run can still proceed, but A0 diagnostic CSVs will be skipped.


## Create oracle answerable dataset and oracle retrieval log


In [9]:
def session_text(session):
    return ' '.join(turn.get('content', '') for turn in session if turn.get('role') == 'user')


answerable_data = []
oracle_retrieval_rows = []
missing_gold_sessions = []
for row in data:
    qid = row['question_id']
    if qid.endswith('_abs') or '_abs' in qid:
        continue
    gold_ids = list(row.get('answer_session_ids') or [])
    if not gold_ids:
        continue

    id_to_session = dict(zip(row['haystack_session_ids'], row['haystack_sessions']))
    id_to_date = dict(zip(row['haystack_session_ids'], row['haystack_dates']))
    ranked_items = []
    for session_id in gold_ids:
        if session_id not in id_to_session:
            missing_gold_sessions.append({'question_id': qid, 'missing_session_id': session_id})
            continue
        ranked_items.append({
            'corpus_id': session_id,
            'text': session_text(id_to_session[session_id]),
            'timestamp': id_to_date.get(session_id),
        })

    if not ranked_items:
        continue

    answerable_data.append(row)
    oracle_row = dict(row)
    oracle_row['retrieval_results'] = {
        'query': row['question'],
        'ranked_items': ranked_items,
        'metrics': {'session': {}, 'turn': {}},
    }
    oracle_retrieval_rows.append(oracle_row)

oracle_dataset_file = A05_DIR / 'longmemeval_s_cleaned_answerable_oracle.json'
oracle_retrieval_log = A05_DIR / 'oracle_retrieval_log.jsonl'
missing_gold_file = A05_DIR / 'oracle_missing_gold_sessions.json'

oracle_dataset_file.write_text(json.dumps(answerable_data, ensure_ascii=False), encoding='utf-8')
with oracle_retrieval_log.open('w', encoding='utf-8') as f:
    for row in oracle_retrieval_rows:
        print(json.dumps(row, ensure_ascii=False), file=f)
missing_gold_file.write_text(json.dumps(missing_gold_sessions, indent=2, ensure_ascii=False), encoding='utf-8')

print('Oracle answerable examples:', len(answerable_data))
print('Oracle retrieval rows:', len(oracle_retrieval_rows))
print('Missing gold sessions:', len(missing_gold_sessions))
print('Oracle dataset:', oracle_dataset_file)
print('Oracle retrieval log:', oracle_retrieval_log)


Oracle answerable examples: 470
Oracle retrieval rows: 470
Missing gold sessions: 0
Oracle dataset: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/longmemeval_s_cleaned_answerable_oracle.json
Oracle retrieval log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_retrieval_log.jsonl


## Run oracle evidence reader


In [10]:
from tqdm.auto import tqdm


def count_lines(path):
    if not path or not Path(path).exists():
        return 0
    with Path(path).open(encoding='utf-8', errors='replace') as f:
        return sum(1 for _ in f)


def count_occurrences(path, needle):
    if not path or not Path(path).exists():
        return 0
    count = 0
    with Path(path).open(encoding='utf-8', errors='replace') as f:
        for line in f:
            if needle in line:
                count += 1
    return count


env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')

oracle_run_id = time.strftime('%Y%m%d-%H%M%S')
oracle_generation_dir = A05_DIR / 'oracle_generation_logs' / GEN_MODEL_ALIAS
oracle_generation_dir.mkdir(parents=True, exist_ok=True)
oracle_suffix = f'_{oracle_run_id}_oracle'
oracle_hyp_file = None
oracle_generation_stdout_log = oracle_generation_dir / f'run_generation{oracle_suffix}.stdout.log'
expected_oracle_examples = count_lines(oracle_retrieval_log)

if RUN_ORACLE_GENERATION:
    if not OPENAI_API_KEY:
        raise RuntimeError('Set Kaggle Secret OPENAI_API_KEY before running oracle generation.')

    cmd = [
        sys.executable, 'run_generation.py',
        '--in_file', str(oracle_retrieval_log),
        '--out_dir', str(oracle_generation_dir),
        '--out_file_suffix', oracle_suffix,
        '--model_name', GEN_MODEL_NAME,
        '--model_alias', GEN_MODEL_ALIAS,
        '--retriever_type', 'flat-session',
        '--merge_key_expansion_into_value', 'none',
        '--topk_context', str(1000),
        '--history_format', HISTORY_FORMAT,
        '--gen_length', str(GEN_LENGTH),
        '--useronly', USERONLY,
        '--cot', 'true',
        '--con', 'false',
    ]
    if OPENAI_BASE_URL:
        cmd.extend(['--openai_base_url', OPENAI_BASE_URL])

    print('$', ' '.join(str(x) for x in cmd))
    print('Writing command output to:', oracle_generation_stdout_log)
    print('Expected oracle examples:', expected_oracle_examples)

    start_time = time.time()
    with oracle_generation_stdout_log.open('w', encoding='utf-8') as stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_DIR / 'src' / 'generation'),
            env=env,
            stdout=stream,
            stderr=subprocess.STDOUT,
            text=True,
        )

        pbar = tqdm(total=expected_oracle_examples, desc='Oracle generation', unit='example')
        last_done = 0
        last_report_ts = 0

        while True:
            hyp_files = sorted(oracle_generation_dir.glob(f'*{oracle_suffix}'), key=lambda p: p.stat().st_mtime)
            if hyp_files:
                oracle_hyp_file = hyp_files[-1]

            successful = count_lines(oracle_hyp_file) if oracle_hyp_file else 0
            failed = count_occurrences(oracle_generation_stdout_log, 'One exception captured')
            done = min(successful + failed, expected_oracle_examples)
            if done > last_done:
                pbar.update(done - last_done)
                last_done = done

            now = time.time()
            if now - last_report_ts >= 30:
                percent = (done / expected_oracle_examples * 100) if expected_oracle_examples else 0
                elapsed = now - start_time
                rate = done / elapsed if elapsed > 0 else 0
                remaining = ((expected_oracle_examples - done) / rate) if rate > 0 else None
                eta = f'{remaining/60:.1f} min' if remaining is not None else 'unknown'
                print(f'Oracle generation progress: {done}/{expected_oracle_examples} ({percent:.1f}%) | success={successful} | failed={failed} | elapsed={elapsed/60:.1f} min | ETA={eta}')
                print('Stdout log:', oracle_generation_stdout_log)
                last_report_ts = now

            if proc.poll() is not None:
                break
            time.sleep(5)

        successful = count_lines(oracle_hyp_file) if oracle_hyp_file else 0
        failed = count_occurrences(oracle_generation_stdout_log, 'One exception captured')
        done = min(successful + failed, expected_oracle_examples)
        if done > last_done:
            pbar.update(done - last_done)
        pbar.close()

    oracle_generation_latency_seconds = time.time() - start_time
    if proc.returncode != 0:
        print('Oracle generation failed. Last log lines:')
        for line in oracle_generation_stdout_log.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]:
            print(line)
        raise RuntimeError(f'run_generation.py failed with exit code {proc.returncode}')

    hyp_files = sorted(oracle_generation_dir.glob(f'*{oracle_suffix}'), key=lambda p: p.stat().st_mtime)
    if hyp_files:
        oracle_hyp_file = hyp_files[-1]
    if not oracle_hyp_file or count_lines(oracle_hyp_file) != expected_oracle_examples:
        raise RuntimeError(f'Oracle generation incomplete. Expected {expected_oracle_examples}, got {count_lines(oracle_hyp_file)}.')

else:
    hyp_files = sorted(oracle_generation_dir.glob('*_oracle'), key=lambda p: p.stat().st_mtime)
    oracle_hyp_file = hyp_files[-1] if hyp_files else None
    oracle_generation_latency_seconds = None
    print('Skipped oracle generation.')

print('Oracle hypothesis file:', oracle_hyp_file)
print('Oracle generation stdout log:', oracle_generation_stdout_log if oracle_generation_stdout_log.exists() else None)
print('Oracle generation latency seconds:', oracle_generation_latency_seconds)


$ /usr/bin/python3 run_generation.py --in_file /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_retrieval_log.jsonl --out_dir /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2 --out_file_suffix _20260528-084019_oracle --model_name cx/gpt-5.2 --model_alias router-gpt-5.2 --retriever_type flat-session --merge_key_expansion_into_value none --topk_context 1000 --history_format json --gen_length 300 --useronly false --cot true --con false --openai_base_url https://splashed-nastily-stopped.ngrok-free.dev/v1
Writing command output to: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/run_generation_20260528-084019_oracle.stdout.log
Expected oracle examples: 470


Oracle generation:   0%|          | 0/470 [00:00<?, ?example/s]

Oracle generation progress: 0/470 (0.0%) | success=0 | failed=0 | elapsed=0.0 min | ETA=unknown
Stdout log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/run_generation_20260528-084019_oracle.stdout.log
Oracle generation progress: 2/470 (0.4%) | success=2 | failed=0 | elapsed=0.5 min | ETA=117.1 min
Stdout log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/run_generation_20260528-084019_oracle.stdout.log
Oracle generation progress: 10/470 (2.1%) | success=10 | failed=0 | elapsed=1.0 min | ETA=46.0 min
Stdout log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/run_generation_20260528-084019_oracle.stdout.log
Oracle generation progress: 17/470 (3.6%) | success=17 | failed=0 | elapsed=1.5 min | ETA=40.0 min
Stdout log: /kaggle/working/LongMemEval-Experiment/results/A05

## Evaluate oracle hypotheses


In [11]:
oracle_eval_file = None
oracle_evaluation_stdout_log = None
oracle_evaluation_latency_seconds = None

if RUN_ORACLE_EVALUATION:
    if not oracle_hyp_file:
        raise RuntimeError('No oracle hypothesis file found. Run oracle generation first or place an existing oracle output in the oracle generation directory.')
    if not OPENAI_API_KEY:
        raise RuntimeError('Set Kaggle Secret OPENAI_API_KEY before running oracle evaluation.')

    cmd = [sys.executable, 'evaluate_qa.py', METRIC_MODEL_SHORT, str(oracle_hyp_file), str(oracle_dataset_file)]
    oracle_eval_file = Path(str(oracle_hyp_file) + f'.eval-results-{METRIC_MODEL_SHORT}')
    oracle_evaluation_stdout_log = Path(str(oracle_hyp_file) + f'.evaluate_qa-{METRIC_MODEL_SHORT}.stdout.log')
    expected_evals = count_lines(oracle_hyp_file)

    if oracle_eval_file.exists():
        oracle_eval_file.unlink()

    print('$', ' '.join(str(x) for x in cmd))
    print('Writing command output to:', oracle_evaluation_stdout_log)
    print('Expected oracle evaluations:', expected_evals)

    start_time = time.time()
    with oracle_evaluation_stdout_log.open('w', encoding='utf-8') as stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_DIR / 'src' / 'evaluation'),
            env=env,
            stdout=stream,
            stderr=subprocess.STDOUT,
            text=True,
        )

        pbar = tqdm(total=expected_evals, desc='Oracle LLM judge', unit='example')
        last_done = 0
        last_report_ts = 0

        while True:
            done = min(count_lines(oracle_eval_file), expected_evals)
            if done > last_done:
                pbar.update(done - last_done)
                last_done = done

            now = time.time()
            if now - last_report_ts >= 30:
                percent = (done / expected_evals * 100) if expected_evals else 0
                elapsed = now - start_time
                rate = done / elapsed if elapsed > 0 else 0
                remaining = ((expected_evals - done) / rate) if rate > 0 else None
                eta = f'{remaining/60:.1f} min' if remaining is not None else 'unknown'
                print(f'Oracle judge progress: {done}/{expected_evals} ({percent:.1f}%) | elapsed={elapsed/60:.1f} min | ETA={eta}')
                print('Evaluation log:', oracle_eval_file)
                print('Stdout log:', oracle_evaluation_stdout_log)
                last_report_ts = now

            if proc.poll() is not None:
                break
            time.sleep(5)

        done = min(count_lines(oracle_eval_file), expected_evals)
        if done > last_done:
            pbar.update(done - last_done)
        pbar.close()

    oracle_evaluation_latency_seconds = time.time() - start_time
    if proc.returncode != 0:
        print('Oracle evaluation failed. Last log lines:')
        for line in oracle_evaluation_stdout_log.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]:
            print(line)
        raise RuntimeError(f'evaluate_qa.py failed with exit code {proc.returncode}')

    if count_lines(oracle_eval_file) != expected_evals:
        raise RuntimeError(f'Oracle evaluation incomplete. Expected {expected_evals}, got {count_lines(oracle_eval_file)}.')
else:
    if oracle_hyp_file:
        matches = sorted(Path(str(oracle_hyp_file)).parent.glob(Path(str(oracle_hyp_file)).name + f'.eval-results-{METRIC_MODEL_SHORT}'))
        oracle_eval_file = matches[-1] if matches else None
    print('Skipped oracle evaluation.')

print('Oracle eval file:', oracle_eval_file)
print('Oracle evaluation stdout log:', oracle_evaluation_stdout_log)
print('Oracle evaluation latency seconds:', oracle_evaluation_latency_seconds)


$ /usr/bin/python3 evaluate_qa.py router-gpt-5.2 /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/longmemeval_s_cleaned_answerable_oracle.json
Writing command output to: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.evaluate_qa-router-gpt-5.2.stdout.log
Expected oracle evaluations: 470


Oracle LLM judge:   0%|          | 0/470 [00:00<?, ?example/s]

Oracle judge progress: 0/470 (0.0%) | elapsed=0.0 min | ETA=unknown
Evaluation log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.eval-results-router-gpt-5.2
Stdout log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.evaluate_qa-router-gpt-5.2.stdout.log
Oracle judge progress: 12/470 (2.6%) | elapsed=0.5 min | ETA=19.1 min
Evaluation log: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned/oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.eval-results-router-gpt-5.2
Stdout log: /kaggle/worki

## Aggregate oracle results and compare with A0


In [12]:
def read_jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def get_eval_type(question_id, ref_row):
    if question_id.endswith('_abs') or '_abs' in question_id:
        return 'abstention'
    return ref_row.get('question_type', 'unknown')


def parse_last_int_after(log_path, prefix):
    if not log_path or not Path(log_path).exists():
        return None
    pattern = re.compile(re.escape(prefix) + r'\s*(\d+)')
    value = None
    for line in Path(log_path).read_text(encoding='utf-8', errors='replace').splitlines():
        match = pattern.search(line)
        if match:
            value = int(match.group(1))
    return value


oracle_summary = None
comparison_rows = []

if oracle_eval_file and Path(oracle_eval_file).exists():
    oracle_eval_rows = read_jsonl(oracle_eval_file)
    oracle_scores_by_type = defaultdict(list)
    for row in oracle_eval_rows:
        ref = ref_by_id[row['question_id']]
        score = 1 if row.get('autoeval_label', {}).get('label') else 0
        oracle_scores_by_type[get_eval_type(row['question_id'], ref)].append(score)

    oracle_total_prompt_tokens = parse_last_int_after(oracle_generation_stdout_log, 'Total prompt tokens:')
    oracle_total_completion_tokens = parse_last_int_after(oracle_generation_stdout_log, 'Total completion tokens:')
    oracle_n = len(oracle_eval_rows)
    oracle_all_scores = [score for scores in oracle_scores_by_type.values() for score in scores]
    oracle_type_acc = {
        key: {
            'n': len(scores),
            'accuracy': sum(scores) / len(scores) if scores else None,
            'failures': len(scores) - sum(scores),
        }
        for key, scores in sorted(oracle_scores_by_type.items())
    }
    oracle_summary = {
        'experiment_id': A05_EXPERIMENT_ID,
        'setting': 'oracle_evidence_sessions_answerable_only',
        'dataset': DATASET_NAME,
        'examples_evaluated': oracle_n,
        'overall_accuracy': sum(oracle_all_scores) / len(oracle_all_scores) if oracle_all_scores else None,
        'task_averaged_accuracy': sum(sum(scores) / len(scores) for scores in oracle_scores_by_type.values() if scores) / len([scores for scores in oracle_scores_by_type.values() if scores]) if oracle_scores_by_type else None,
        'accuracy_by_eval_type': oracle_type_acc,
        'reader_model': GEN_MODEL_NAME,
        'judge_model': oracle_eval_rows[0]['autoeval_label']['model'] if oracle_eval_rows else METRIC_MODEL_NAME,
        'total_prompt_tokens': oracle_total_prompt_tokens,
        'total_completion_tokens': oracle_total_completion_tokens,
        'avg_prompt_tokens': None if oracle_total_prompt_tokens is None or not oracle_n else oracle_total_prompt_tokens / oracle_n,
        'avg_completion_tokens': None if oracle_total_completion_tokens is None or not oracle_n else oracle_total_completion_tokens / oracle_n,
        'generation_latency_seconds': oracle_generation_latency_seconds,
        'evaluation_latency_seconds': oracle_evaluation_latency_seconds,
        'paths': {
            'oracle_dataset': str(oracle_dataset_file),
            'oracle_retrieval_log': str(oracle_retrieval_log),
            'oracle_hypotheses': str(oracle_hyp_file),
            'oracle_eval_log': str(oracle_eval_file),
        },
    }
    (A05_DIR / 'oracle_summary.json').write_text(json.dumps(oracle_summary, indent=2, ensure_ascii=False), encoding='utf-8')

    print('Oracle overall accuracy:', round(oracle_summary['overall_accuracy'], 4))
    print('Oracle task-averaged accuracy:', round(oracle_summary['task_averaged_accuracy'], 4))
    print('Oracle avg prompt tokens:', oracle_summary['avg_prompt_tokens'])
    print('Oracle by eval type:')
    for eval_type, item in oracle_type_acc.items():
        print(f"  {eval_type}: {item['accuracy']:.4f} ({item['n']})")
else:
    print('No oracle eval file found; skipping oracle aggregation.')

a0_summary = None
if HAS_A0_ARTIFACTS and 'summary' in a0_paths:
    a0_summary = json.loads(Path(a0_paths['summary']).read_text(encoding='utf-8'))

if a0_summary and oracle_summary:
    eval_types = sorted(set(a0_summary.get('accuracy_by_eval_type', {}).keys()) | set(oracle_summary.get('accuracy_by_eval_type', {}).keys()))
    for eval_type in eval_types:
        a0_item = a0_summary.get('accuracy_by_eval_type', {}).get(eval_type, {})
        oracle_item = oracle_summary.get('accuracy_by_eval_type', {}).get(eval_type, {})
        comparison_rows.append({
            'eval_type': eval_type,
            'a0_n': a0_item.get('n'),
            'a0_accuracy': a0_item.get('accuracy'),
            'oracle_n': oracle_item.get('n'),
            'oracle_accuracy': oracle_item.get('accuracy'),
            'oracle_minus_a0': None if a0_item.get('accuracy') is None or oracle_item.get('accuracy') is None else oracle_item.get('accuracy') - a0_item.get('accuracy'),
        })

    comparison = {
        'a0_experiment_id': A0_EXPERIMENT_ID,
        'a05_experiment_id': A05_EXPERIMENT_ID,
        'a0_overall_accuracy': a0_summary.get('overall_accuracy'),
        'oracle_overall_accuracy': oracle_summary.get('overall_accuracy'),
        'oracle_gap_overall': oracle_summary.get('overall_accuracy') - a0_summary.get('overall_accuracy'),
        'a0_avg_prompt_tokens': a0_summary.get('cost_latency', {}).get('avg_prompt_tokens'),
        'oracle_avg_prompt_tokens': oracle_summary.get('avg_prompt_tokens'),
        'by_eval_type': comparison_rows,
    }
    (A05_DIR / 'oracle_vs_a0_summary.json').write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')

    print('\nA0 vs Oracle')
    print('A0 overall:', round(comparison['a0_overall_accuracy'], 4))
    print('Oracle overall:', round(comparison['oracle_overall_accuracy'], 4))
    print('Oracle gap:', round(comparison['oracle_gap_overall'], 4))
    for row in comparison_rows:
        if row['oracle_accuracy'] is not None:
            print(f"{row['eval_type']}: A0={row['a0_accuracy']} Oracle={row['oracle_accuracy']:.4f} Gap={row['oracle_minus_a0']:.4f}")


Oracle overall accuracy: 0.917
Oracle task-averaged accuracy: 0.9158
Oracle avg prompt tokens: 8688.597872340426
Oracle by eval type:
  knowledge-update: 0.8889 (72)
  multi-session: 0.8430 (121)
  single-session-assistant: 1.0000 (56)
  single-session-preference: 0.8333 (30)
  single-session-user: 0.9688 (64)
  temporal-reasoning: 0.9606 (127)


## Standardize A0 retrieval metrics


In [13]:
def dcg(relevances):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances))


def retrieval_metrics_for_row(row, topks=REPORT_TOPKS):
    ranked_ids = [item.get('corpus_id') for item in row.get('retrieval_results', {}).get('ranked_items', [])]
    gold_ids = set(row.get('answer_session_ids') or [])
    metrics = {}
    if not gold_ids:
        for k in topks:
            metrics[f'hit_any@{k}'] = None
            metrics[f'hit_all@{k}'] = None
            metrics[f'gold_fraction@{k}'] = None
            metrics[f'ndcg@{k}'] = None
        metrics['mrr'] = None
        return metrics

    first_rank = None
    for idx, corpus_id in enumerate(ranked_ids, start=1):
        if corpus_id in gold_ids:
            first_rank = idx
            break
    metrics['mrr'] = 0.0 if first_rank is None else 1.0 / first_rank
    for k in topks:
        top_ids = ranked_ids[:k]
        hits = [corpus_id for corpus_id in top_ids if corpus_id in gold_ids]
        gains = [1 if corpus_id in gold_ids else 0 for corpus_id in top_ids]
        ideal = [1] * min(len(gold_ids), k)
        ideal_dcg = dcg(ideal)
        metrics[f'hit_any@{k}'] = 1.0 if hits else 0.0
        metrics[f'hit_all@{k}'] = 1.0 if len(set(hits)) == len(gold_ids) else 0.0
        metrics[f'gold_fraction@{k}'] = len(set(hits)) / len(gold_ids)
        metrics[f'ndcg@{k}'] = 0.0 if ideal_dcg == 0 else dcg(gains) / ideal_dcg
    return metrics


if HAS_A0_ARTIFACTS and 'retrieval_log' in a0_paths:
    a0_retrieval_rows = read_jsonl(a0_paths['retrieval_log'])
    standardized_records = []
    by_eval_type = defaultdict(lambda: defaultdict(list))
    overall = defaultdict(list)

    for row in a0_retrieval_rows:
        qid = row['question_id']
        ref = ref_by_id.get(qid, row)
        eval_type = get_eval_type(qid, ref)
        if eval_type == 'abstention':
            continue
        metrics = retrieval_metrics_for_row(row)
        record = {
            'question_id': qid,
            'eval_type': eval_type,
            'question_type': ref.get('question_type'),
            'num_gold_sessions': len(row.get('answer_session_ids') or []),
        }
        record.update(metrics)
        standardized_records.append(record)
        for key, value in metrics.items():
            if value is not None:
                by_eval_type[eval_type][key].append(value)
                overall[key].append(value)

    summary = {
        'population': 'non_abstention_answerable_examples',
        'n': len(standardized_records),
        'overall': {key: sum(values) / len(values) for key, values in sorted(overall.items()) if values},
        'by_eval_type': {
            eval_type: {key: sum(values) / len(values) for key, values in sorted(metric_map.items()) if values}
            for eval_type, metric_map in sorted(by_eval_type.items())
        },
    }

    std_json = A05_DIR / 'A0_retrieval_metrics_standardized.json'
    std_csv = A05_DIR / 'A0_retrieval_metrics_standardized.csv'
    std_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    with std_csv.open('w', newline='', encoding='utf-8') as f:
        fieldnames = list(standardized_records[0].keys()) if standardized_records else []
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(standardized_records)

    print('Standardized retrieval metrics written:', std_json)
    print('Population n:', summary['n'])
    for key in ['hit_any@5', 'hit_all@5', 'gold_fraction@5', 'mrr', 'ndcg@5']:
        print(key, round(summary['overall'].get(key), 4))
else:
    print('Skipping standardized retrieval metrics because A0 retrieval log is unavailable.')


Skipping standardized retrieval metrics because A0 retrieval log is unavailable.


## Failed-case taxonomy and manual audit sample


In [14]:
def auto_error_labels(eval_type, correct, gold_ids, retrieved_ids, topk=TOPK_CONTEXT):
    if correct:
        return []
    labels = []
    gold_set = set(gold_ids or [])
    top_ids = retrieved_ids[:topk]
    top_gold = set(top_ids) & gold_set

    if eval_type == 'abstention':
        return ['abstention_false_answer']
    if gold_set and not top_gold:
        labels.append('retrieval_miss')
    elif gold_set and top_gold != gold_set:
        labels.append('partial_evidence')
    elif gold_set:
        labels.append('reader_failure')
    else:
        labels.append('no_gold_session_annotation')

    if 'reader_failure' in labels or 'partial_evidence' in labels:
        if eval_type == 'multi-session':
            labels.append('aggregation_or_synthesis_error')
        elif eval_type == 'temporal-reasoning':
            labels.append('temporal_filter_or_reasoning_error')
        elif eval_type == 'single-session-preference':
            labels.append('preference_inference_error')
    return labels


if HAS_A0_ARTIFACTS and {'eval_log', 'retrieval_log'}.issubset(a0_paths.keys()):
    a0_eval_rows = read_jsonl(a0_paths['eval_log'])
    a0_retrieval_by_id = {row['question_id']: row for row in read_jsonl(a0_paths['retrieval_log'])}
    taxonomy_rows = []
    label_counter = Counter()
    eval_type_counter = Counter()

    for row in a0_eval_rows:
        qid = row['question_id']
        ref = ref_by_id[qid]
        eval_type = get_eval_type(qid, ref)
        correct = bool(row.get('autoeval_label', {}).get('label'))
        retrieval_row = a0_retrieval_by_id.get(qid, {})
        gold_ids = list(retrieval_row.get('answer_session_ids') or ref.get('answer_session_ids') or [])
        retrieved_ids = [item.get('corpus_id') for item in retrieval_row.get('retrieval_results', {}).get('ranked_items', [])]
        top_ids = retrieved_ids[:TOPK_CONTEXT]
        num_gold_retrieved = len(set(top_ids) & set(gold_ids))
        labels = auto_error_labels(eval_type, correct, gold_ids, retrieved_ids)
        if not correct:
            eval_type_counter[eval_type] += 1
            label_counter.update(labels)
        taxonomy_rows.append({
            'question_id': qid,
            'eval_type': eval_type,
            'question_type': ref.get('question_type'),
            'correct': correct,
            'question': ref.get('question'),
            'gold_answer': ref.get('answer'),
            'hypothesis': row.get('hypothesis', ''),
            'judge_label': row.get('autoeval_label', {}).get('label'),
            'retrieval_miss': 'retrieval_miss' in labels,
            'retrieved_session_ids_topk': '|'.join(str(x) for x in top_ids),
            'gold_answer_session_ids': '|'.join(str(x) for x in gold_ids),
            'num_gold_sessions': len(gold_ids),
            'num_gold_sessions_retrieved_topk': num_gold_retrieved,
            'error_labels': '|'.join(labels),
            'manual_error_labels': '',
            'manual_notes': '',
        })

    failed_taxonomy_rows = [row for row in taxonomy_rows if not row['correct']]
    taxonomy_csv = A05_DIR / 'A0_failed_cases_diagnostic.csv'
    with taxonomy_csv.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(taxonomy_rows[0].keys()))
        writer.writeheader()
        writer.writerows(failed_taxonomy_rows)

    taxonomy_summary = {
        'num_failed_cases': len(failed_taxonomy_rows),
        'failures_by_eval_type': dict(eval_type_counter),
        'auto_error_label_counts': dict(label_counter),
    }
    taxonomy_summary_file = A05_DIR / 'A0_failed_case_taxonomy_summary.json'
    taxonomy_summary_file.write_text(json.dumps(taxonomy_summary, indent=2, ensure_ascii=False), encoding='utf-8')

    sample_limits = {
        'multi-session': 20,
        'temporal-reasoning': 20,
        'single-session-preference': 10,
        'abstention': 7,
    }
    audit_sample = []
    for eval_type, limit in sample_limits.items():
        rows = [row for row in failed_taxonomy_rows if row['eval_type'] == eval_type]
        audit_sample.extend(rows[:limit])
    audit_csv = A05_DIR / 'A0_failed_case_taxonomy_sample57.csv'
    with audit_csv.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(taxonomy_rows[0].keys()))
        writer.writeheader()
        writer.writerows(audit_sample)

    print('Failed taxonomy CSV:', taxonomy_csv)
    print('Manual audit sample CSV:', audit_csv)
    print('Failures by eval type:', dict(eval_type_counter))
    print('Auto label counts:', dict(label_counter))
else:
    print('Skipping failed-case taxonomy because A0 eval/retrieval logs are unavailable.')


Skipping failed-case taxonomy because A0 eval/retrieval logs are unavailable.


## Completion length diagnostic


In [15]:
def get_tokenizer():
    try:
        import tiktoken
        return tiktoken.get_encoding('o200k_base')
    except Exception:
        return None


def approx_token_count(text, tokenizer=None):
    text = text or ''
    if tokenizer is not None:
        return len(tokenizer.encode(text, allowed_special={'<|endoftext|>'}))
    return max(1, len(text.split()))


def percentile(values, p):
    if not values:
        return None
    values = sorted(values)
    idx = (len(values) - 1) * p
    lo = math.floor(idx)
    hi = math.ceil(idx)
    if lo == hi:
        return values[int(idx)]
    return values[lo] * (hi - idx) + values[hi] * (idx - lo)


if HAS_A0_ARTIFACTS and 'eval_log' in a0_paths:
    tokenizer = get_tokenizer()
    eval_rows = read_jsonl(a0_paths['eval_log'])
    lengths = []
    no_final = 0
    hit_max = 0
    diagnostic_rows = []
    for row in eval_rows:
        hyp = row.get('hypothesis', '')
        n_tokens = approx_token_count(hyp, tokenizer)
        lengths.append(n_tokens)
        lowered = hyp.lower()
        has_finalish_answer = any(marker in lowered for marker in ['answer:', 'final answer', 'therefore', 'so the answer'])
        if not has_finalish_answer:
            no_final += 1
        if n_tokens >= GEN_LENGTH - 5:
            hit_max += 1
        diagnostic_rows.append({
            'question_id': row['question_id'],
            'eval_type': get_eval_type(row['question_id'], ref_by_id[row['question_id']]),
            'correct': bool(row.get('autoeval_label', {}).get('label')),
            'approx_completion_tokens': n_tokens,
            'hit_max_tokens_estimate': n_tokens >= GEN_LENGTH - 5,
            'no_final_answer_marker': not has_finalish_answer,
        })

    completion_summary = {
        'n': len(lengths),
        'completion_tokens_min': min(lengths) if lengths else None,
        'completion_tokens_p50': percentile(lengths, 0.50),
        'completion_tokens_p90': percentile(lengths, 0.90),
        'completion_tokens_p95': percentile(lengths, 0.95),
        'completion_tokens_p99': percentile(lengths, 0.99),
        'completion_tokens_max': max(lengths) if lengths else None,
        'hit_max_tokens_rate_estimate': hit_max / len(lengths) if lengths else None,
        'no_final_answer_marker_rate': no_final / len(lengths) if lengths else None,
        'tokenizer': 'o200k_base' if tokenizer else 'whitespace_fallback',
    }
    completion_json = A05_DIR / 'A0_completion_length_diagnostic.json'
    completion_csv = A05_DIR / 'A0_completion_length_diagnostic.csv'
    completion_json.write_text(json.dumps(completion_summary, indent=2, ensure_ascii=False), encoding='utf-8')
    with completion_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(diagnostic_rows[0].keys()))
        writer.writeheader()
        writer.writerows(diagnostic_rows)
    print(json.dumps(completion_summary, indent=2))
    print('Completion diagnostic files:', completion_json, completion_csv)
else:
    print('Skipping completion length diagnostic because A0 eval log is unavailable.')


Skipping completion length diagnostic because A0 eval log is unavailable.


## Prompt leakage check


In [16]:
leakage_report = A05_DIR / 'A0_prompt_leakage_check.md'
gen_source = (REPO_DIR / 'src' / 'generation' / 'run_generation.py').read_text(encoding='utf-8')

checks = []
checks.append(('reader_input_file_contains_gold_answer_field', True, 'The retrieval/eval input rows contain `answer`, but this is metadata in the file.'))
checks.append(('prepare_prompt_formats_gold_answer', 'answer_prompt_template.format(history_string, question_date_string, question_string)' in gen_source, 'Expected prompt format uses history, question date, and question only.'))
checks.append(('prepare_prompt_reads_entry_answer', "entry['answer']" in gen_source[gen_source.find('def prepare_prompt'):gen_source.find('@backoff')], 'Should be False. `prepare_prompt` should not read the gold answer.'))
checks.append(('generation_prints_gold_answer_to_stdout', "'answer': entry['answer']" in gen_source, 'The script prints gold answer to stdout for logging; this is not inserted into the prompt.'))

lines = [
    '# A0 Prompt Leakage Check',
    '',
    'This is a static check over `src/generation/run_generation.py`.',
    '',
    '| Check | Value | Note |',
    '|---|---:|---|',
]
for name, value, note in checks:
    lines.append(f'| `{name}` | `{value}` | {note} |')

lines.extend([
    '',
    'Interpretation:',
    '',
    '- The generation input file may contain `answer` because the upstream retrieval log preserves metadata.',
    '- The reader prompt should not format or expose that field.',
    '- The judge step is allowed to see gold answers because it evaluates hypotheses.',
])
leakage_report.write_text('\n'.join(lines), encoding='utf-8')
print(leakage_report.read_text(encoding='utf-8'))


# A0 Prompt Leakage Check

This is a static check over `src/generation/run_generation.py`.

| Check | Value | Note |
|---|---:|---|
| `reader_input_file_contains_gold_answer_field` | `True` | The retrieval/eval input rows contain `answer`, but this is metadata in the file. |
| `prepare_prompt_formats_gold_answer` | `True` | Expected prompt format uses history, question date, and question only. |
| `prepare_prompt_reads_entry_answer` | `False` | Should be False. `prepare_prompt` should not read the gold answer. |
| `generation_prints_gold_answer_to_stdout` | `True` | The script prints gold answer to stdout for logging; this is not inserted into the prompt. |

Interpretation:

- The generation input file may contain `answer` because the upstream retrieval log preserves metadata.
- The reader prompt should not format or expose that field.
- The judge step is allowed to see gold answers because it evaluates hypotheses.


## Package A0.5 outputs


In [17]:
archive = ROOT / f'{A05_EXPERIMENT_ID}_outputs.tar.gz'
run_cmd(['tar', '-czf', str(archive), 'results'], cwd=REPO_DIR, env=env, check=False)
print('Archive:', archive)
print('A0.5 output directory:', A05_DIR)
print('Files:')
for path in sorted(A05_DIR.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(A05_DIR), '|', round(path.stat().st_size / 1024, 1), 'KB')


$ tar -czf /kaggle/working/A05_oracle_diagnostic_lme_s_cleaned_outputs.tar.gz results
Archive: /kaggle/working/A05_oracle_diagnostic_lme_s_cleaned_outputs.tar.gz
A0.5 output directory: /kaggle/working/LongMemEval-Experiment/results/A05_oracle_diagnostic_lme_s_cleaned
Files:
 - A0_prompt_leakage_check.md | 0.9 KB
 - longmemeval_s_cleaned_answerable_oracle.json | 236499.1 KB
 - oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle | 381.7 KB
 - oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.eval-results-router-gpt-5.2 | 408.3 KB
 - oracle_generation_logs/router-gpt-5.2/oracle_retrieval_log.jsonl_testlog_top1000context_jsonformat_useronlyfalse_20260528-0840_20260528-084019_oracle.evaluate_qa-router-gpt-5.2.stdout.log | 492.0 KB
 - oracle_generation_logs/router-gpt-5.2/run_generation_20260528-0